# Capítulo 5 · Actividad: ¿cuánto consume este auto?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/02_actividad_mpg_regresion.ipynb)

Entrena una **regresión** que prediga el rendimiento de un auto en **millas por galón** (`mpg`) y consigue el mejor modelo que puedas. Haz todo el proceso que vimos en clase; el preprocesamiento lo decides tú.

| Variable | Significado |
|---|---|
| `mpg` | **Target**: millas por galón (más alto = consume menos) |
| `cylinders` | Número de cilindros |
| `displacement` | Cilindrada del motor (pulgadas cúbicas) |
| `horsepower` | Potencia (caballos de fuerza) |
| `weight` | Peso (libras) |
| `acceleration` | Segundos para ir de 0 a 60 mph |
| `model_year` | Año del modelo (70 = 1970) |
| `origin` | Región de origen: `usa`, `europe` o `japan` |
| `name` | Nombre del auto |

Como mínimo, tu notebook debe incluir:

- Un **modelo lineal base**: coeficientes, métricas de error y $R^2$.
- Los mismos modelos que vimos en clase: regresión simple, lineal múltiple, Ridge, Lasso, polinomial y RANSAC.
- Una **tabla final** comparando los modelos con validación cruzada y test, y tus **conclusiones**: ¿qué modelo elegirías y por qué?

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`

In [ ]:
import pandas as pd
import numpy as np

# Cargar los datos. Si estás en la carpeta workspace/, la ruta relativa suele ser hacia atrás:
ruta = '../data/mpg.csv' 
# Si falla, puedes usar la ruta absoluta que tenías antes: '/home/anyeloot/ml-course/data/mpg.csv'

df = pd.read_csv(ruta)

# Revisar la información general para detectar nulos y tipos de datos
df.info()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Configurar un estilo visual limpio
sns.set_theme(style="whitegrid")

# ---------------------------------------------------------
# GRÁFICA 1: Distribución de la variable objetivo (mpg)
# ---------------------------------------------------------
plt.figure(figsize=(8, 4))
# Si aún no existe un dataframe limpio, usamos una copia del original
# para que el resto del notebook pueda seguir sin errores.
if 'df_clean' not in globals():
    df_clean = df.copy()

sns.histplot(df_clean['mpg'], bins=30, kde=True, color='teal')
plt.title('Distribución del Rendimiento (Millas por Galón - mpg)')
plt.xlabel('Millas por galón (mpg)')
plt.ylabel('Frecuencia')
plt.show()

# ---------------------------------------------------------
# GRÁFICA 2: Relación de variables físicas vs el rendimiento
# ---------------------------------------------------------
# Un pairplot nos deja ver cómo interactúan las variables más importantes
columnas_clave = ['mpg', 'weight', 'horsepower', 'displacement']
sns.pairplot(df_clean[columnas_clave], diag_kind='kde', corner=True, plot_kws={'alpha':0.6})
plt.suptitle('Relación entre Consumo, Peso, Potencia y Cilindrada', y=1.02)
plt.show()

# ---------------------------------------------------------
# GRÁFICA 3: Mapa de calor de correlaciones
# ---------------------------------------------------------
plt.figure(figsize=(10, 8))
correlaciones = df_clean.select_dtypes(include="number").corr()
# Dibujamos el mapa de calor con los valores numéricos
sns.heatmap(correlaciones, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Mapa de Correlación entre Variables')
plt.show()

In [ ]:
# 1. Eliminar la columna 'name' (el nombre del auto no sirve para predecir el consumo)
df_clean = df.drop(columns=['name'])

# 2. Arreglar 'horsepower' (a veces viene como texto o trae un "?" en los datos faltantes)
# Esto fuerza a que sea número y convierte los errores en nulos (NaN)
df_clean['horsepower'] = pd.to_numeric(df_clean['horsepower'], errors='coerce')

# Como los nulos en horsepower son muy pocos, la forma más rápida y segura es eliminar esas filas
df_clean = df_clean.dropna()

# 3. Convertir la columna de texto 'origin' (usa, europe, japan) en columnas numéricas (One-Hot Encoding)
# Usamos drop_first=True para evitar multicolinealidad, algo clave en Regresión Lineal
df_clean = pd.get_dummies(df_clean, columns=['origin'], drop_first=True, dtype=int)

# Ver el dataset final limpio y 100% numérico
display(df_clean.head())
display(df_clean.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# 1. Separar variables independientes (X) de la variable objetivo (y)
X = df_clean.drop(columns=['mpg'])
y = df_clean['mpg']

# 2. Dividir en conjunto de entrenamiento (80%) y prueba (20%)
# random_state=42 asegura que siempre se dividan igual los datos (útil para que tus resultados no cambien)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Datos de entrenamiento: {X_train.shape[0]} filas")
print(f"Datos de prueba: {X_test.shape[0]} filas\n")

# 3. Entrenar el Modelo Lineal Base
modelo_base = LinearRegression()
modelo_base.fit(X_train, y_train)

# 4. Hacer predicciones con los datos de prueba
y_pred_base = modelo_base.predict(X_test)

# 5. Calcular métricas de error
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
mae_base = mean_absolute_error(y_test, y_pred_base)
r2_base = r2_score(y_test, y_pred_base)

print("--- MÉTRICAS DEL MODELO LINEAL BASE ---")
print(f"Error Cuadrático Medio (RMSE): {rmse_base:.2f} mpg")
print(f"Error Absoluto Medio (MAE): {mae_base:.2f} mpg")
print(f"Coeficiente de Determinación (R²): {r2_base:.4f}\n")

# 6. Mostrar los coeficientes (qué tanto afecta cada variable al consumo)
import pandas as pd
coeficientes = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': modelo_base.coef_
}).sort_values(by='Coeficiente', ascending=False)

print("--- COEFICIENTES DEL MODELO ---")
display(coeficientes)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar el tamaño del lienzo para tener dos gráficas lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---------------------------------------------------------
# GRÁFICA 1: Valores Reales vs. Predicciones
# ---------------------------------------------------------
axes[0].scatter(y_test, y_pred_base, alpha=0.7, color='dodgerblue', edgecolor='k')
# Dibujar la línea ideal donde la predicción es exactamente igual al valor real
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_title('Valores Reales vs. Predicciones (Modelo Base)')
axes[0].set_xlabel('Valores Reales (mpg)')
axes[0].set_ylabel('Predicciones del Modelo (mpg)')

# ---------------------------------------------------------
# GRÁFICA 2: Análisis de Residuos (Errores)
# ---------------------------------------------------------
# El residuo es la diferencia entre el valor real y lo que el modelo predijo
residuos = y_test - y_pred_base

axes[1].scatter(y_pred_base, residuos, alpha=0.7, color='darkorchid', edgecolor='k')
# Dibujar la línea del cero error
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_title('Gráfica de Residuos')
axes[1].set_xlabel('Predicciones (mpg)')
axes[1].set_ylabel('Residuo (Error en mpg)')

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# GRÁFICA 3 (Opcional): Importancia de los coeficientes
# ---------------------------------------------------------
plt.figure(figsize=(8, 4))
sns.barplot(x='Coeficiente', y='Variable', data=coeficientes, palette='vlag')
plt.title('Impacto de cada variable en el rendimiento (mpg)')
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

# Calcular MAPE (Mean Absolute Percentage Error)
mape_base = mean_absolute_percentage_error(y_test, y_pred_base)

print("--- RENDIMIENTO PORCENTUAL DEL MODELO ---")
print(f"🎯 Nivel de explicación (R²): {r2_base * 100:.2f}%")
print(f"⚠️ Margen de error promedio (MAPE): {mape_base * 100:.2f}%")

# Ejemplo práctico de cómo interpretar el error
ejemplo_mpg = 30
falla_estimada = ejemplo_mpg * mape_base
print(f"\nEjemplo: Si un auto realmente rinde {ejemplo_mpg} mpg,")
print(f"tu modelo calculará un valor con un margen de error de +/- {falla_estimada:.2f} mpg.")

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RANSACRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

# 1. Función para evaluar cualquier modelo y sacar las 4 columnas de tu imagen
def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_te, y_te):
    # Entrenar con tus datos
    modelo.fit(X_tr, y_tr)
    
    # Calcular validación cruzada (CV) en entrenamiento
    r2_cv = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring='r2').mean()
    
    # Hacer predicciones en prueba (Test)
    y_pred = modelo.predict(X_te)
    
    # Calcular métricas finales
    return {
        'Modelo': nombre,
        'R² (CV)': r2_cv,
        'R² (Test)': r2_score(y_te, y_pred),
        'MAE (Test)': mean_absolute_error(y_te, y_pred),
        'MSE (Test)': mean_squared_error(y_te, y_pred)
    }

# 2. Construir la tabla usando los modelos de la actividad
resultados = [
    # Para regresión simple, extraemos solo la columna 'weight'
    evaluar_modelo('Regresión Simple (weight)', LinearRegression(), X_train[['weight']], y_train, X_test[['weight']], y_test),
    # Para los demás, usamos todas las variables (tu modelo base)
    evaluar_modelo('Regresión Lineal Múltiple', LinearRegression(), X_train, y_train, X_test, y_test),
    evaluar_modelo('Ridge (Regularización L2)', Ridge(alpha=1.0), X_train, y_train, X_test, y_test),
    evaluar_modelo('Lasso (Selección de Features L1)', Lasso(alpha=0.1), X_train, y_train, X_test, y_test),
    evaluar_modelo('Polinomial Grado 2 (Mejor ajuste)', make_pipeline(PolynomialFeatures(degree=2), LinearRegression()), X_train, y_train, X_test, y_test),
    evaluar_modelo('RANSAC Regressor', RANSACRegressor(random_state=42), X_train, y_train, X_test, y_test)
]

# 3. Imprimir con el formato exacto
tabla_final = pd.DataFrame(resultados)
print("=== RESULTADOS CON DECISIONES BASADAS EN DATOS Y CORRELACIÓN ===")
# float_format="%.4f" fuerza los 4 decimales y justify='right' alinea las columnas
print(tabla_final.to_string(index=False, float_format="%.4f", justify='right'))